# CodeT5-small Fine-tuning on Colab GPU
Run each cell top to bottom. Make sure you set **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# Cell 1 — Confirm GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers==4.40.2 accelerate sentencepiece

In [ ]:
# Cell 3 — Upload your train.csv and val.csv
# When prompted, select BOTH files from: training/t5/data/train.csv and val.csv
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
uploaded = files.upload()   # pick train.csv AND val.csv

for fname, data in uploaded.items():
    dest = f'/content/data/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved {fname} -> {dest}')

In [ ]:
# Cell 4 — Training (with intent context in input)
import logging, os, random, sys, time
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)

CONFIG = {
    'model_name':                 'Salesforce/codet5-base',
    'save_dir':                   '/content/saved_model',
    'checkpoint_dir':             '/content/checkpoints',
    'train_file':                 '/content/data/train.csv',
    'val_file':                   '/content/data/val.csv',
    'max_input_length':           256,
    'max_target_length':          128,
    'batch_size':                 16,
    'gradient_accumulation_steps': 1,
    'num_epochs':                 10,
    'learning_rate':              5e-5,
    'warmup_steps':               200,
    'seed':                       42,
    'weight_decay':               0.01,
    'early_stopping_patience':    2,
    'prefix':                     '',   # intent already in input_text e.g. "fix binary operator error: ..."
}

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)],
)

class BugFixDataset(Dataset):
    def __init__(self, dataframe, tokenizer, config):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.config = config

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_text  = self.config['prefix'] + str(self.data.iloc[idx]['input_text'])
        target_text = str(self.data.iloc[idx]['target_text'])
        inputs  = self.tokenizer(input_text,  max_length=self.config['max_input_length'],
                                  padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(target_text, max_length=self.config['max_target_length'],
                                  padding='max_length', truncation=True, return_tensors='pt')
        labels = targets['input_ids'].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            'input_ids':      inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels':         labels,
        }

def evaluate(model, val_loader, tokenizer, device, config):
    model.eval()
    total_loss, n_batches, exact_matches, total_gen = 0.0, 0, 0, 0
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbl  = batch['labels'].to(device)
            loss = model(input_ids=ids, attention_mask=mask, labels=lbl).loss
            total_loss += loss.item(); n_batches += 1
            if i < 50:
                gen = model.generate(input_ids=ids, attention_mask=mask,
                                     max_length=config['max_target_length'])
                lbl_dec = lbl.clone(); lbl_dec[lbl_dec == -100] = tokenizer.pad_token_id
                preds = tokenizer.batch_decode(gen, skip_special_tokens=True)
                tgts  = tokenizer.batch_decode(lbl_dec, skip_special_tokens=True)
                for p, t in zip(preds, tgts):
                    exact_matches += int(p.strip() == t.strip()); total_gen += 1
    model.train()
    avg_loss = total_loss / max(n_batches, 1)
    em_pct   = (exact_matches / total_gen * 100) if total_gen else 0.0
    return avg_loss, em_pct

def train():
    random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed'])
    torch.manual_seed(CONFIG['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logging.info(f'Using device: {device}')
    os.makedirs(CONFIG['save_dir'], exist_ok=True)
    os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

    train_df = pd.read_csv(CONFIG['train_file'])
    val_df   = pd.read_csv(CONFIG['val_file'])
    logging.info(f'Train rows: {len(train_df)} | Val rows: {len(val_df)}')
    logging.info(f'Sample input: {train_df["input_text"].iloc[0]}')

    logging.info('Loading tokenizer and model ...')
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], use_fast=False)
    model     = T5ForConditionalGeneration.from_pretrained(CONFIG['model_name']).to(device)
    logging.info(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    train_loader = DataLoader(BugFixDataset(train_df, tokenizer, CONFIG),
                              batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
    val_loader   = DataLoader(BugFixDataset(val_df,   tokenizer, CONFIG),
                              batch_size=CONFIG['batch_size'], num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    opt_steps   = (len(train_loader) // CONFIG['gradient_accumulation_steps']) * CONFIG['num_epochs']
    scheduler   = get_linear_schedule_with_warmup(optimizer,
                      num_warmup_steps=CONFIG['warmup_steps'], num_training_steps=opt_steps)

    best_val_loss = float('inf')
    patience_counter = 0
    start = time.time()
    global_step = 0

    for epoch in range(CONFIG['num_epochs']):
        logging.info(f"--- EPOCH {epoch+1}/{CONFIG['num_epochs']} ---")
        model.train()
        epoch_loss = 0.0

        for step, batch in enumerate(train_loader):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbl  = batch['labels'].to(device)
            loss = model(input_ids=ids, attention_mask=mask, labels=lbl).loss
            (loss / CONFIG['gradient_accumulation_steps']).backward()

            if (step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                global_step += 1

            epoch_loss += loss.item()

            if step % 50 == 0:
                elapsed = time.time() - start
                eta = (elapsed / max(global_step, 1)) * (opt_steps - global_step) / 3600
                logging.info(f'Epoch {epoch+1} | Step {step}/{len(train_loader)} '
                             f'| Loss: {loss.item():.4f} | ETA: {eta:.1f}h')

        val_loss, em = evaluate(model, val_loader, tokenizer, device, CONFIG)
        avg_train = epoch_loss / len(train_loader)
        logging.info(f'Epoch {epoch+1} done | Train Loss: {avg_train:.4f} '
                     f'| Val Loss: {val_loss:.4f} | Exact Match: {em:.1f}%')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            model.save_pretrained(CONFIG['save_dir'])
            tokenizer.save_pretrained(CONFIG['save_dir'])
            logging.info(f'Best model saved (val_loss={val_loss:.4f})')
        else:
            patience_counter += 1
            logging.info(f'No improvement. Patience: {patience_counter}/{CONFIG["early_stopping_patience"]}')
            if patience_counter >= CONFIG['early_stopping_patience']:
                logging.info('Early stopping triggered.')
                break

    logging.info(f'Training complete! Best val loss: {best_val_loss:.4f}')

train()

In [ ]:
# Cell 5 — Zip and download the trained model
import shutil
shutil.make_archive('/content/saved_model', 'zip', '/content/saved_model')
from google.colab import files
files.download('/content/saved_model.zip')
print('Download started — check your browser downloads.')